In [1]:
# Packages to be imported
#
# This notebook implements a Cobaya-based Bayesian analysis of BOSS DR12 NGC
# power-spectrum data, with a nodal reconstruction of the primordial spectrum
# P_R(k) using CAMB for the underlying cosmological calculations.

import cobaya  # Library for Bayesian analysis in cosmology
import sys     # System-specific parameters and functions (path handling etc.)

# Append the local installation path of CAMB (Code for Anisotropies in the Microwave Background)
# so that the 'camb' Python module can be imported correctly.
sys.path.append('/Users/guillermo/Desktop/code/CAMB')

import camb                     # CAMB: code for computing CMB and matter power spectra
import numpy as np              # NumPy: numerical computations and array handling
import sympy                    # SymPy: symbolic mathematics (not heavily used here)
import math                     # Standard mathematical functions from the Python library
import matplotlib.pyplot as plt # Matplotlib: plotting library
from scipy.signal import find_peaks  # Utility to identify maxima/minima in spectra

# Import specific interpolation and integration utilities from SciPy.
from scipy.interpolate import CubicSpline  # Cubic spline interpolation for smooth approximations
from scipy.interpolate import interp1d     # 1D interpolation function
import scipy.integrate as integrate       # Numerical integration routines

# Cobaya script for nodal reconstruction of the primordial scalar power spectrum \(P_R(k)\)

This notebook implements a Cobaya-based Bayesian analysis of BOSS DR12 NGC
galaxy power-spectrum data, reconstructing the primordial scalar power spectrum
\(P_R(k)\) using a nodal parameterisation combined with a BOSS NGC likelihood
for the windowed galaxy monopole power spectrum \(P_g(k)\).

In [29]:
# Selection of the BOSS NGC effective redshift bin and the number of primordial nodes.
#
# The analysis can be performed at different effective redshifts (z_eff) for
# the BOSS NGC sample. Here we choose the z-bin and the number of nodes that
# parameterise the primordial spectrum.

# Select the z-bin of BOSS NGC data:
zbin = 1  # NGC_z1 (effective redshift z_eff ≈ 0.38)
# zbin = 2 # NGC_z3 (effective redshift z_eff ≈ 0.61)

# Select the number of primordial nodes (must be >= 2; 2 corresponds to a pure power law).
number_knots = 4

# Cosmological parameters #

In [31]:
# Cosmological constants and fiducial cosmology.
#
# We adopt a fiducial cosmology consistent with the Planck 2018 TT,TE,EE+lowE+lensing results,
# and we treat a subset of parameters as fixed in this reconstruction.

# Cosmological constants
c = 2.99792458e5  # Speed of light in km/s
H = 1 / (c / 100) # Hubble constant normalization factor

# Fiducial cosmology, assumed equal to Planck 2018 TT,TE,EE+lowE+lensing.

# Assumptions that will not be sampled:
Omegak_BOSS = 0      # Curvature density parameter (assumed spatially flat universe)
k_pivot = 0.05       # Pivot scale in Mpc⁻¹ for the primordial spectrum

# Core BOSS cosmology parameters:
sigma8_BOSS = 0.8111         # RMS matter fluctuation amplitude at 8 Mpc/h
n_s_BOSS = 0.9649            # Scalar spectral index
h_BOSS = 0.6736              # Dimensionless Hubble parameter
Omega_b_BOSS_h2 = 0.02237    # Baryon density parameter Ω_b h²
Omegam_BOSS = 0.3153         # Total matter density parameter Ω_m
Omega_nu_BOSS_h2 = 0.0       # Neutrino density parameter Ω_ν h² (treated as radiation)

# Parameters that we indirectly compute or assume to match Planck values:
A_s_BOSS = 2.098903E-9       # Scalar amplitude A_s (converted from ln(10¹⁰ A_s))
Omega_CDM_BOSS_h2 = 0.1200   # Cold dark matter density parameter Ω_CDM h²
tau_BOSS = 0.0544            # Optical depth to reionization (Planck 2018 value)
mnu_BOSS = 0.00              # Sum of neutrino masses corresponding to Ω_ν = 0
num_nu_massless_BOSS = 3.046 # Effective number of neutrino species (Planck 2018 value)

# Planck power-law equivalent knot parameters for the primordial spectrum:
y1_BOSS = -19.901252666476413 # Log-amplitude at the first knot (correlated with A_s, n_s)
yN_BOSS = -20.030289987352592 # Log-amplitude at the last knot (correlated with A_s, n_s)

# Indirect cosmological parameters derived from the above:
H0_BOSS = h_BOSS * 100                            # Hubble constant in km/s/Mpc
Omega_b_BOSS = Omega_b_BOSS_h2 / h_BOSS**2        # Baryon density parameter Ω_b
Omega_CDM_BOSS = Omega_CDM_BOSS_h2 / h_BOSS**2    # Cold dark matter density Ω_CDM
Omega_nu_BOSS = Omega_nu_BOSS_h2 / h_BOSS**2      # Neutrino density parameter Ω_ν
Omega_Lambda_BOSS = 1 - (Omega_b_BOSS + Omega_CDM_BOSS + Omega_nu_BOSS + Omegak_BOSS
)  # Dark energy density parameter Ω_Λ

# LSS parameters #

In [32]:
# Large-scale structure (LSS) primordial power-law templates.
#
# These helper functions provide a standard power-law primordial spectrum
# with and without explicit h factors in the definition of the wavenumber k.

# Power-law primordial power spectrum; k is defined in units of h/Mpc.
def PrimordialPowerLaw(As, ns, k):
    """
    Primordial scalar power spectrum with power-law form in h-units:

        P_R(k) = A_s (k / (k_pivot / h_BOSS))^(n_s - 1)

    Parameters
    ----------
    As : float
        Scalar amplitude A_s.
    ns : float
        Scalar spectral index n_s.
    k : float or np.ndarray
        Wavenumber in units of h/Mpc.
    """
    return As * (k / (k_pivot / h_BOSS))**(ns - 1)

# Power-law primordial power spectrum without h in the k units.
def PrimordialPowerLawSinh(As, ns, k):
    """
    Primordial scalar power spectrum with power-law form in physical units:

        P_R(k) = A_s (k / k_pivot)^(n_s - 1)

    Parameters
    ----------
    As : float
        Scalar amplitude A_s.
    ns : float
        Scalar spectral index n_s.
    k : float or np.ndarray
        Wavenumber in units of Mpc⁻¹ (no explicit h factor).
    """
    return As * (k / k_pivot)**(ns - 1)

In [33]:
# Fixed linear galaxy bias parameter.
#
# In this analysis the linear bias is set to a fixed value b = 1.0, which
# is fully degenerate with the overall amplitude in the model and is thus
# not sampled.

b = 1.0

# k and z binning #

In [34]:
# k- and z-binning for the BOSS NGC analysis.
#
# We define the k-array (in units of h/Mpc) and the effective redshifts
# for the BOSS NGC bins, including an optional z = 0 bin.

# k-array definition and rebinning.
#
# We rebin at 40 k-scales, defined in h/Mpc, and we use this array throughout
# the sampling and in the construction of the windowed power-spectrum model.
KhArrayBOSS = np.array([
    0.00747 , 0.016038, 0.025638, 0.035455, 0.045362, 0.055292,
    0.065254, 0.07521 , 0.08519 , 0.095171, 0.10516 , 0.115138,
    0.125136, 0.135121, 0.145112, 0.155106, 0.165101, 0.175095,
    0.185092, 0.195084, 0.205079, 0.215077, 0.225071, 0.235069,
    0.245066, 0.255065, 0.265064, 0.275061, 0.285058, 0.295058,
    0.305055, 0.315053, 0.325052, 0.33505 , 0.345049, 0.355047,
    0.365046, 0.375043, 0.385045, 0.395041
])

# When we cut at k = 0.3 h/Mpc, we obtain a reduced array length for the data vector.
DimKhArrayBOSSCut03 = len(KhArrayBOSS) - 10
KhCutOff03 = 0.3  # Cut-off scale in h/Mpc used for the likelihood

# z-binning: effective redshifts of each NGC bin.
zBOSSz1 = 0.383891  # Effective redshift z_eff for NGC z1
zBOSSz3 = 0.60594   # Effective redshift z_eff for NGC z3

# We add a z = 0 bin (which can be useful in some intermediate calculations).
zBOSS = np.array([0, zBOSSz1, zBOSSz3])

# Reading BOSS P(k) data, window functions and covariance matrices

In this section we read the monopole galaxy power-spectrum data, the survey
window-function matrices and the covariance matrices for the BOSS DR12 NGC z1
sample, and store them in a structured dictionary for use in the likelihood.

In [35]:
def read_data(path_to_data):
    """
    Read and preprocess BOSS DR12 NGC z1 power spectrum, covariance matrix, and
    window function data.

    This routine constructs a structured dictionary containing:
      - 'data_pk' : binned galaxy monopole power spectrum P_g(k) up to k = 0.3 h/Mpc
      - 'cov'     : covariance matrix for the binned P_g(k) (30 × 30 submatrix)
      - 'inv_cov' : inverse covariance matrix (computed if the covariance is non-zero)
      - 'window'  : window-function matrix mapping an input P_g(k) on 40 k-bins
                    to the observed 30 k-bins used in the likelihood

    Parameters
    ----------
    path_to_data : str
        Base directory containing the BOSS DR12 NGC z1 data files.

    Returns
    -------
    dict
        Dictionary with the keys 'data_pk', 'cov', 'inv_cov', and 'window', each
        storing the corresponding arrays with the z-bin index already selected
        via the global variable `zbin`.
    """
    data = {}

    # Define file paths for BOSS NGC z1 data (simulated Pk, window function and covariance).
    simulated_pk_z1_filename = (
        path_to_data
        + 'BOSS_DR12_NGC_z1/Data_P0P2_30KBins_BOSS_DR12_NGC_z1.txt'
    )
    window_function_z1_filename = (
        path_to_data
        + 'BOSS_DR12_NGC_z1/'
        + 'W_BOSS_DR12_NGC_z1_V6C_1_1_1_1_1_10_200_200_averaged_v1.matrix'
    )
    covariance_matrix_z1_filename = (
        path_to_data
        + 'BOSS_DR12_NGC_z1/'
        + 'C_2048_BOSS_DR12_NGC_z1_V6C_1_1_1_1_1_10_200_200_prerecon.txt'
    )

    # Allocate arrays for:
    #  - monopole P_g(k) (only the first 30 k-bins up to k = 0.3 h/Mpc)
    #  - window function (30 × 40 matrix)
    #  - covariance matrix (30 × 30 submatrix)
    data['data_pk'] = np.zeros((len(zBOSS), DimKhArrayBOSSCut03))
    data['window'] = np.zeros((len(zBOSS), DimKhArrayBOSSCut03, len(KhArrayBOSS)))
    data['cov'] = np.zeros((len(zBOSS), DimKhArrayBOSSCut03, DimKhArrayBOSSCut03))

    # Read the simulated monopole galaxy power spectrum P_g(k) for the chosen z-bin.
    with open(simulated_pk_z1_filename) as file:
        file.readline()  # Skip header line
        for i in range(DimKhArrayBOSSCut03):
            line = file.readline().split()
            data['data_pk'][zbin][i] = float(line[1])

    # Read the covariance matrix and compute its inverse.
    try:
        with open(covariance_matrix_z1_filename, 'r') as file:
            for i in range(DimKhArrayBOSSCut03):
                line = file.readline().strip()

                if not line:
                    continue

                parts = line.split()

                if len(parts) < DimKhArrayBOSSCut03:
                    print(f"Warning: Line {i+1} has insufficient data: {line}")
                    continue

                for j in range(DimKhArrayBOSSCut03):
                    data['cov'][zbin][i][j] = float(parts[j])

        if np.any(
            data['cov'][zbin][:DimKhArrayBOSSCut03, :DimKhArrayBOSSCut03] != 0
        ):
            data['inv_cov'] = np.linalg.inv(
                data['cov'][zbin][:DimKhArrayBOSSCut03, :DimKhArrayBOSSCut03]
            )
        else:
            data['inv_cov'] = np.zeros_like(
                data['cov'][zbin][:DimKhArrayBOSSCut03, :DimKhArrayBOSSCut03]
            )

    except FileNotFoundError:
        print(f"Error: File not found at {covariance_matrix_z1_filename}")
    except ValueError as e:
        print(f"Error: Could not convert data to float in covariance matrix: {e}")

    # Read the window-function matrix W(k_obs, k_true).
    try:
        with open(window_function_z1_filename, 'r') as file:
            for i in range(DimKhArrayBOSSCut03):
                line = file.readline().strip()

                if not line:
                    continue

                parts = line.split()

                if len(parts) < len(KhArrayBOSS):
                    print(f"Warning: Line {i+1} has insufficient data: {line}")
                    continue

                for j in range(len(KhArrayBOSS)):
                    data['window'][zbin][i][j] = float(parts[j])

    except FileNotFoundError:
        print(f"Error: File not found at {window_function_z1_filename}")
    except ValueError as e:
        print(f"Error: Could not convert data to float in window function: {e}")

    return data

# Read data using the specified path and output the available keys.
data = read_data('/Users/guillermo/Desktop/BOSS_Data/')
data.keys()

dict_keys(['data_pk', 'window', 'cov', 'inv_cov'])

# Cobaya theory and likelihood interface classes

We now define:
- a Cobaya `Theory` class (`NodesInPrimordialPk`) to implement the nodal primordial spectrum, and
- a Cobaya external `Likelihood` class (`Pklike`) to model the BOSS NGC galaxy monopole and compute its likelihood.

In [2]:
# Cobaya interface base classes.
#
# We create:
#   - a Cobaya theory class `NodesInPrimordialPk` for the nodal primordial spectrum, and
#   - an external likelihood class `Pklike` for the BOSS galaxy power-spectrum.
from cobaya.theory import Theory
from cobaya.likelihood import Likelihood

In [37]:
# Theory class implementing the nodal modification of the primordial power spectrum P_R(k).
class NodesInPrimordialPk(Theory):
    """
    Cobaya theory component that parameterises the primordial scalar power spectrum
    using a set of nodes in (log k, log P_R) and provides:
      - a nodal reconstruction evaluated on KhArrayBOSS, and
      - a linear log-log fit mimicking a power-law primordial spectrum.

    The primordial spectra are exposed to CAMB/Cobaya via dictionaries with keys
    'kmin', 'kmax', 'k', 'Pk', and 'log_regular'.
    """

    def initialize(self):
        """
        Initialise the theory object with the reference k-array and the number of nodes.
        """
        # Reference k-array (in h/Mpc) on which the primordial spectrum is evaluated.
        self.ks = KhArrayBOSS

        # Set the number of nodes for this run (must be >= 2).
        self.number_knots = number_knots  # Or any other integer ≥ 2

    # Definition of the knot parameters to be sampled and how they enter the primordial spectrum.
    def calculate(self, state, want_derived=True, **params_values_dict):
        """
        Build the nodal primordial scalar power spectrum and its power-law mimic
        given the sampled node positions {x_i} and amplitudes {y_i}.

        Parameters
        ----------
        state : dict
            Cobaya state dictionary to be populated with the primordial spectra.
        want_derived : bool
            Flag indicating whether to compute derived quantities (unused).
        params_values_dict : dict
            Dictionary of parameter values for the x_i and y_i node parameters.
        """
        number_knots = self.number_knots

        # Only works for n >= 2 (at least two nodes are required).
        if number_knots < 2:
            raise ValueError("number_knots must be >= 2")

        # Ordering algorithm for node positions in the normalised interval [0, 1].
        # It affects intermediate nodes (not the outer ones).
        number_knots_red = number_knots - 2

        megacube = np.zeros(number_knots)

        # First node at x1.
        megacube[0] = params_values_dict['x1']

        # Last node at x_N.
        megacube[-1] = params_values_dict[f'x{number_knots}']

        # Compute intermediate megacube values (x2, ..., x_{N-1}) ensuring ordering.
        for i in range(1, number_knots - 1):  # Exclude 0 and maximum number of nodes
            xi = params_values_dict[f'x{i+1}']  # The first x_i will be x2
            megacube[i] = megacube[i-1] + (1 - megacube[i-1]) * (
                1 - (1 - xi)**(1 / (number_knots_red - (i-1)))
            )

        # ---------------------------------------------------------------------
        # Define the k-PPS nodes in log k-space.
        # ---------------------------------------------------------------------

        # We start with the k knots:

        # First, we add the knot corresponding to x1 (minimum k).
        nodes_logk = [np.log(KhArrayBOSS[0])]

        # Add knots at intermediate x values, mapped between k_min and k_cut = 0.3 h/Mpc.
        for i in range(1, number_knots - 1):
            nodes_logk.append(
                np.log(KhArrayBOSS[0]) +
                megacube[i] * (np.log(KhCutOff03) - np.log(KhArrayBOSS[0]))
            )

        # Finally, add the x_N knot at the maximum k value.
        nodes_logk.append(np.log(KhArrayBOSS[-1]))

        # ---------------------------------------------------------------------
        # Define the y-knots (logarithmic amplitudes of the primordial spectrum).
        # ---------------------------------------------------------------------

        # Just the y_i values, interpreted as log P_R(k) at the corresponding nodes.
        nodes_logPPS = [params_values_dict[f'y{i+1}'] for i in range(number_knots)]

        # nodes_logk and nodes_logPPS are linearly interpolated in log space.
        # For outer values, the function is extrapolated.
        NodesInterpFunc_nodes = interp1d(
            nodes_logk,
            nodes_logPPS,
            kind='linear',
            fill_value='extrapolate'
        )

        # Construct a modified primordial spectrum P_R(k) evaluated at our k-array.
        # The units must be without h in the limits of k and CAMB/Cobaya expects
        # the dictionary format used below (kmin, kmax, k, Pk, log_regular).
        state['primordial_knot_function_scalar_pk'] = {
            'kmin': KhArrayBOSS[0] * h_BOSS,
            'kmax': KhArrayBOSS[-1] * h_BOSS,
            'k': KhArrayBOSS * h_BOSS,
            'Pk': np.exp(NodesInterpFunc_nodes(np.log(KhArrayBOSS))),
            'log_regular': False
        }

        # ---------------------------------------------------------------------
        # Linear fit in (log k, log P_R) space to build a power-law mimic.
        # ---------------------------------------------------------------------

        # Linear (degree-1) fit in (nodes_logk, nodes_logPPS).
        coeffs = np.polyfit(nodes_logk, nodes_logPPS, deg=1)

        # Define the mimicking function as a 1D polynomial in log k.
        Outer_NodesInterpFunc_nodes = np.poly1d(coeffs)

        # Construct a primordial spectrum evaluated at our interpolated mimic power law.
        state['primordial_scalar_pk'] = {
            'kmin': KhArrayBOSS[0] * h_BOSS,
            'kmax': KhArrayBOSS[-1] * h_BOSS,
            'k': KhArrayBOSS * h_BOSS,
            'Pk': np.exp(Outer_NodesInterpFunc_nodes(np.log(KhArrayBOSS))),
            'log_regular': False
        }

    # Name the modified primordial power spectrum as 'primordial_scalar_pk' to be called in Cobaya.
    def get_primordial_scalar_pk(self):
        """
        Return the power-law–mimic primordial scalar power spectrum dictionary.
        """
        return self.current_state['primordial_scalar_pk']

    # Name the nodal spectrum as 'primordial_knot_function_scalar_pk' to be called in Cobaya.
    def get_primordial_knot_function_scalar_pk(self):
        """
        Return the nodal primordial scalar power spectrum reconstruction dictionary.
        """
        return self.current_state['primordial_knot_function_scalar_pk']

    # Function that contains the parameters to be sampled.
    def get_can_support_params(self):
        """
        Return the list of node parameters (x_i and y_i) that this theory class
        supports and that can be sampled by Cobaya.

        Returns
        -------
        list of str
            Parameter names ['x1', ..., 'xN', 'y1', ..., 'yN'] corresponding
            to node positions and amplitudes.
        """
        # Use the number of nodes defined in the class (default to 2 if not set).
        number_knots = getattr(self, "number_knots", 2)

        # Automatically generate the list of x parameters: x1, x2, ..., xN.
        x_params = [f"x{i+1}" for i in range(number_knots)]

        # Automatically generate the list of y parameters: y1, y2, ..., yN.
        y_params = [f"y{i+1}" for i in range(number_knots)]

        # Return the list of parameters to be sampled by Cobaya.
        return x_params + y_params

In [38]:
# Class incorporating the model (monopole galaxy power spectrum) and the likelihood.
class Pklike(Likelihood):
    """
    External Cobaya likelihood for the BOSS DR12 NGC monopole galaxy power spectrum P_g(k).

    The class:
      - reads BOSS data (P_g(k), covariance and window function) via `read_data`,
      - requests CAMB matter power spectra, primordial spectra and nuisance parameters
        from the Cobaya theory provider,
      - constructs a parametric model for the windowed P_g(k) monopole,
      - evaluates a Gaussian log-likelihood using the inverse covariance matrix.
    """

    def initialize(self):
        """
        Initialise the likelihood by reading the BOSS data and setting up the redshift grid.
        """
        # Path in which the data are stored; we call read_data with this path.
        self.data = read_data('/Users/guillermo/Desktop/BOSS_Data/')
        # self.data = read_data('/Users/guillermo/Desktop/BOSS_Data/')

        # Grid of z to be employed (effective redshifts for NGC bins).
        self.z_win = zBOSS

    def get_requirements(self):
        """
        Specify which cosmological quantities and power spectra are required from the theory.

        Returns
        -------
        dict
            Requirements dictionary interpreted by Cobaya.
        """
        # Cobaya parameters and functions that we use or sample.
        return {
            'omegam': None,
            # "b": None,
            'Pk_interpolator': {
                'z': self.z_win,
                'k_max': 677.7,
                'nonlinear': False,
                'vars_pairs': ([['delta_tot', 'delta_tot']])
            },
            # 'b': None,
            'sigma_s': None,
            'sigma_nl': None,
            'a1': None,
            'a2': None,
            'a3': None,
            'alpha': None,
            # 'Pk_grid': {'z': self.z_win, 'k_max': 0.4, 'nonlinear': False,
            #             'vars_pairs': ([['delta_tot', 'delta_tot']])},
            'CAMBdata': None,
            'primordial_knot_function_scalar_pk': None
        }

    # We define the quantity to compare with data in the likelihood:
    # the windowed galaxy power spectrum, evaluated at 30 k-bins.
    def GalaxyPSWindowed(self, **params_dic):
        """
        Construct the windowed galaxy monopole power spectrum P_g(k) at the selected
        BOSS NGC redshift bin using the parametric model and apply the survey window
        function to match the data vector.
        """
        # Options for printing arrays with enough decimal precision (for debugging).
        np.set_printoptions(precision=24, suppress=True)

        # Parameters that we sample; we need to specify them here.
        sigma_s = self.provider.get_param("sigma_s")
        sigma_nl = self.provider.get_param("sigma_nl")
        a1 = self.provider.get_param("a1")
        a2 = self.provider.get_param("a2")
        a3 = self.provider.get_param("a3")
        alpha = self.provider.get_param("alpha")

        # CAMB results within Cobaya.
        resultsCobaya = self.provider.get_CAMBdata()

        # ---------------------------------------------------------------------
        # DEFINITION OF PRIMORDIAL POWER SPECTRUM
        # ---------------------------------------------------------------------

        # Power-law–like primordial power spectrum and its matter power spectrum (the one we use):
        # This is the power-law–like primordial power spectrum P(k) evaluated at KhArrayBOSS.
        primordialCobaya = self.provider.get_primordial_scalar_pk()
        primordialCobayaArray = np.array(primordialCobaya['Pk'])

        # Construction of linear P_matter(k) from the Cobaya interpolator.
        # This uses the power-law mimic.
        pkCobaya = self.provider.get_Pk_interpolator(
            ('delta_tot', 'delta_tot'),
            nonlinear=False
        )

        # Knot primordial power spectrum evaluated at KhArrayBOSS (only used in Delta Feature).
        primordialKnotFunctionCobaya = self.provider.get_primordial_knot_function_scalar_pk()
        primordialKnotFunctionCobayaArray = np.array(primordialKnotFunctionCobaya['Pk'])

        # Delta Feature definition. For 2 knots, it is zero by construction.
        DeltaPFeature = (
            primordialKnotFunctionCobayaArray - primordialCobayaArray
        ) / primordialCobayaArray

        # Print the two primordial spectra for inspection.
        print(primordialKnotFunctionCobayaArray)
        print(primordialCobayaArray)

        # DeltaPFeature_interp = interp1d(KhArrayBOSS, DeltaPFeatureRaw, kind='cubic',
        #                                 bounds_error=False, fill_value=0)
        # DeltaPFeature = DeltaPFeature_interp(KhArrayBOSS)

        # ---------------------------------------------------------------------
        # P NO-WIGGLE IMPLEMENTATION.
        # We do this by interpolating a maximum/minimum curve and include the O_lin term.
        # ---------------------------------------------------------------------

        # Range to compute wiggles and flatten factor.
        KhArrayBOSSNoWiggle = KhArrayBOSS  # We might extend the range of k to perform smoothing.
        FlattenFactor = 1.75

        # Compute the gradient of the flattened spectrum.
        gradient = np.gradient(
            KhArrayBOSSNoWiggle**FlattenFactor
            * pkCobaya.P(self.z_win[zbin], KhArrayBOSSNoWiggle * h_BOSS),
            KhArrayBOSSNoWiggle
        )

        # Find k-scales corresponding to maxima and minima of the gradient.
        maxima_indices, _ = find_peaks(gradient)
        maxima_kh = KhArrayBOSSNoWiggle[maxima_indices]

        minima_indices, _ = find_peaks(-gradient)
        minima_kh = KhArrayBOSSNoWiggle[minima_indices]

        int_order = 2  # Polynomial interpolation order

        Pk_max = pkCobaya.P(self.z_win[zbin], maxima_kh * h_BOSS)
        Pk_min = pkCobaya.P(self.z_win[zbin], minima_kh * h_BOSS)

        # Interpolation of the maxima and minima spectra in flattened space.
        max_interp_fun = interp1d(
            maxima_kh,
            maxima_kh**FlattenFactor * Pk_max,
            kind=int_order,
            fill_value="extrapolate"
        )
        min_interp_fun = interp1d(
            minima_kh,
            minima_kh**FlattenFactor * Pk_min,
            kind=int_order,
            fill_value="extrapolate"
        )

        interp_max = max_interp_fun(KhArrayBOSSNoWiggle)
        interp_min = min_interp_fun(KhArrayBOSSNoWiggle)

        Interp_Maxima = np.array(interp_max)
        Interp_Minima = np.array(interp_min)

        # Mean interpolation between maxima and minima.
        Interp_Mean = (Interp_Maxima + Interp_Minima) / 2

        Interp_Mean_functions = []  # Placeholder list (not used further but kept for consistency).

        Interp_Mean_function = interp1d(
            KhArrayBOSSNoWiggle,
            1 / KhArrayBOSSNoWiggle**FlattenFactor * Interp_Mean,
            kind=int_order,
            fill_value="extrapolate"
        )

        exp_factor = 3500000000  # Exponential factor controlling the transition to no-wiggle

        # Interpolate the mean function in KhArrayBOSS and KhArrayBOSS/alpha.
        PmNoWiggleFunctionLinearBOSS = Interp_Mean_function(KhArrayBOSS)
        PmNoWiggleFunctionLinearBOSSAlpha = Interp_Mean_function(KhArrayBOSS / alpha)

        # Transition from the matter power spectrum to the no-wiggle one.
        PmNoWiggleCorrectedFunctionLinearBOSSVector = (
            (1 - np.exp(-exp_factor * KhArrayBOSS**8))
            * PmNoWiggleFunctionLinearBOSS
            + np.exp(-exp_factor * KhArrayBOSS**8)
            * pkCobaya.P(self.z_win[zbin], KhArrayBOSS * h_BOSS)
        )

        PmNoWiggleCorrectedFunctionLinearBOSSVectorAlpha = (
            (1 - np.exp(-exp_factor * (KhArrayBOSS / alpha)**8))
            * PmNoWiggleFunctionLinearBOSSAlpha
            + np.exp(-exp_factor * (KhArrayBOSS / alpha)**8)
            * pkCobaya.P(self.z_win[zbin], (KhArrayBOSS / alpha) * h_BOSS)
        )

        # Define O_lin only evaluated at KhArrayBOSS/alpha.
        OlinAlpha = (
            pkCobaya.P(self.z_win[zbin], (KhArrayBOSS / alpha) * h_BOSS)
            / PmNoWiggleCorrectedFunctionLinearBOSSVectorAlpha
            - 1
        )

        # ---------------------------------------------------------------------
        # GALAXY POWER SPECTRUM PARAMETRIC MODEL
        # ---------------------------------------------------------------------

        # Fingers-of-God effect functional.
        F = 1 / (1 + KhArrayBOSS**2 * sigma_s**2 / 2)

        # Broadband functional A(k).
        A = a1 / KhArrayBOSS**2 + a2 / KhArrayBOSS + a3

        # Galaxy power-spectrum model with P_no-wiggle(k) incorporating F(k) and A(k).
        PgNw = F * b**2 * (h_BOSS**3 * PmNoWiggleCorrectedFunctionLinearBOSSVector) + A

        # O_lin and DeltaFeature model with non-linear BAO damping.
        Pg = PgNw * (
            1
            + (OlinAlpha + DeltaPFeature + OlinAlpha * DeltaPFeature)
            * np.exp(-KhArrayBOSS**2 * sigma_nl**2 / 2)
        )

        # Apply the proper window function (30×40), to obtain a 30-element Pg(k) windowed array
        # from the input 40 Pg(k) array.
        PgWindowed = np.dot(data['window'][zbin], Pg)  # (30x40) * (1x40)

        # Return the final value of the windowed monopole galaxy power spectrum
        # at our k-array and selected z-bin.
        return PgWindowed

    # Likelihood calculation.
    def logp(self, **params_values):
        """
        Compute the log-likelihood for the windowed monopole galaxy power-spectrum model
        given the BOSS DR12 NGC z1 data and covariance.

        The likelihood is assumed Gaussian in P_g(k), with covariance estimated from
        mocks, and is returned as -χ²/2.
        """
        # Allocate arrays for the monopole values.
        PMonopoleBineado = np.zeros((len(zBOSS), DimKhArrayBOSSCut03))

        # PMonopoleBineado is equal to the values given by GalaxyPSWindowed.
        PMonopoleBineado[zbin] = self.GalaxyPSWindowed(**params_values)

        # Construct the Gaussian likelihood as a χ² with the inverse covariance term.
        # Compute the difference vector.
        delta_P = data['data_pk'][zbin] - PMonopoleBineado[zbin]  # Shape (30,)

        # Compute the chi-squared term: (delta_P^T) * C_inv * (delta_P).
        Chi2 = np.dot(delta_P, np.dot(data['inv_cov'], delta_P))  # (1x30) * (30x30) * (30x1)

        # Return -lnlike / 2 as expected by Cobaya.
        return -Chi2 / 2

In [39]:
# Standard deviations obtained from Planck 2018 analysis.
Omega_b_BOSS_h2_std = 0.000145746
Omega_CDM_BOSS_h2_std = 0.00117818
H0_BOSS_std = 0.535407
y1_BOSS_std = 0.0151341
yN_BOSS_std = 0.0164396

# Creation of the multivariate Gaussian that will be used as a cosmological prior.
#
# Mean values of the multivariate Gaussian for each dimension.
mean_vector = [Omega_b_BOSS_h2, Omega_CDM_BOSS_h2, H0_BOSS, y1_BOSS, yN_BOSS]

# Covariance matrix for Planck TTTEEE low-ℓ lowE lensing.
# Order: Ω_b h², Ω_CDM h², H0, y1, yN.
PlanckDR3_base_plikHM_TTTEEE_lowl_lowE_lensing_y1yNTransformed_covMatrix = np.array([
    [ 2.12418149e-08, -9.03204492e-08,  5.50402459e-05, -4.88216602e-08,  8.25376070e-07],
    [-9.03204492e-08,  1.38810729e-06, -6.02340614e-04,  3.37529611e-06, -8.67212270e-06],
    [ 5.50402459e-05, -6.02340614e-04,  2.86660525e-01, -1.29089585e-03,  4.15331943e-03],
    [-4.88216602e-08,  3.37529611e-06, -1.29089585e-03,  2.29039585e-04,  1.33058461e-04],
    [ 8.25376070e-07, -8.67212270e-06,  4.15331943e-03,  1.33058461e-04,  2.70261875e-04]
])

# Components of the manual multivariate Gaussian to be created.
determinant = np.linalg.det(PlanckDR3_base_plikHM_TTTEEE_lowl_lowE_lensing_y1yNTransformed_covMatrix)

# Inverse matrix for the Gaussian prior.
if determinant != 0:
    inverse_matrix = np.linalg.inv(PlanckDR3_base_plikHM_TTTEEE_lowl_lowE_lensing_y1yNTransformed_covMatrix)
else:
    print("The matrix is singular and does not have an inverse.")

def ResidualsVector(ombh2, omch2, H0, y1, y4):
    """
    Compute residuals with respect to the Planck-based fiducial values for
    (Ω_b h², Ω_CDM h², H0, y1, yN).

    Returns
    -------
    np.ndarray
        Residual vector of length 5.
    """
    return np.array([
        ombh2 - Omega_b_BOSS_h2,
        omch2 - Omega_CDM_BOSS_h2,
        H0 - H0_BOSS,
        y1 - y1_BOSS,
        y4 - yN_BOSS
    ])

# Construction of the (log) normalized multivariate Gaussian PDF.
def multivariate_gaussian_pdf(ombh2, omch2, H0, y1, y4):
    """
    Evaluate the (log) multivariate Gaussian prior based on the Planck DR3
    covariance matrix for the cosmological parameters and knot amplitudes.

    Parameters
    ----------
    ombh2, omch2, H0 : float
        Baryon density, cold dark matter density and Hubble constant.
    y1, y4 : float
        Log-amplitudes of the first and last primordial nodes.
    """
    return np.log(
        (2 * np.pi)**(-3/2)
        * determinant**(-1/2)
        * np.exp(
            -0.5 * np.dot(
                np.transpose(ResidualsVector(ombh2, omch2, H0, y1, y4)),
                np.dot(inverse_matrix, ResidualsVector(ombh2, omch2, H0, y1, y4))
            )
        )
    )

In [40]:
# Creation of the multivariate Gaussian that will be used as prior for the model nuisance parameters.
# Mean values of the multivariate Gaussian for each dimension.
b_prior = 1.701883
sigma_s_prior = 2.496557
sigma_nl_prior = 11.797564
a1_prior = -4.473341
a2_prior = 823.973292
a3_prior = -570.034840
alpha_prior = 0.982907

b_prior_std = 0.234316
sigma_s_prior_std = 1.690473
sigma_nl_prior_std = 3.542338
a1_prior_std = 3.670959
a2_prior_std = 466.675762
a3_prior_std = 831.033986
alpha_prior_std = 0.1

mean_vector_model = [sigma_s_prior, sigma_nl_prior, a1_prior, a2_prior, a3_prior]

# Covariance matrix according to the Patchy-MD mocks 1–2000 of NGC z1.
CovMat_NGC_z1_MocksFrom1To2000 = np.array([
    [ 2.857698e+00,  9.257194e-02, -5.292273e+00,  6.949390e+02, -1.033725e+03],
    [ 9.257194e-02,  1.254816e+01,  5.353935e-01, -7.065998e+01,  1.335037e+02],
    [-5.292273e+00,  5.353935e-01,  1.347594e+01, -1.491606e+03,  2.459528e+03],
    [ 6.949390e+02, -7.065998e+01, -1.491606e+03,  2.177863e+05, -3.698259e+05],
    [-1.033725e+03,  1.335037e+02,  2.459528e+03, -3.698259e+05,  6.906175e+05]
])

# Components of the manual multivariate Gaussian to be created.
determinantModel = np.linalg.det(CovMat_NGC_z1_MocksFrom1To2000)

# Inverse matrix for the model prior.
if determinantModel != 0:
    inverse_matrix_model = np.linalg.inv(CovMat_NGC_z1_MocksFrom1To2000)
else:
    print("The matrix is singular and does not have an inverse.")

def ResidualsVectorModel(sigma_s, sigma_nl, a1, a2, a3):
    """
    Compute residuals with respect to the Patchy-MD mock-based best-fit
    values for the nuisance parameters of the galaxy power-spectrum model.

    Returns
    -------
    np.ndarray
        Residual vector of length 5.
    """
    return np.array([
        sigma_s - sigma_s_prior,
        sigma_nl - sigma_nl_prior,
        a1 - a1_prior,
        a2 - a2_prior,
        a3 - a3_prior,
    ])

# Construction of the (log) normalized multivariate Gaussian PDF for the model.
def multivariate_gaussian_pdf_model(sigma_s, sigma_nl, a1, a2, a3):
    """
    Evaluate the (log) multivariate Gaussian prior for the nuisance parameters
    (σ_s, σ_nl, a1, a2, a3) using the covariance estimated from BOSS NGC z1
    Patchy-MD mocks.

    Parameters
    ----------
    sigma_s, sigma_nl : float
        Small-scale damping and non-linear BAO damping parameters.
    a1, a2, a3 : float
        Coefficients of the broadband A(k) term.
    """
    return np.log(
        (2 * np.pi)**(-3/2)
        * determinantModel**(-1/2)
        * np.exp(
            -0.5 * np.dot(
                np.transpose(ResidualsVectorModel(sigma_s, sigma_nl, a1, a2, a3)),
                np.dot(inverse_matrix_model, ResidualsVectorModel(sigma_s, sigma_nl, a1, a2, a3))
            )
        )
    )

In [41]:
# Dictionary given to Cobaya. We need to specify:
#  - Likelihood (class including the likelihood and the model),
#  - Theory class (in which we modify the primordial power spectrum to have knots),
#  - Parameters (fixed and to be sampled),
#  - Sampler to be used and its specifications,
#  - Output path where results will be saved,
#  - Additional options such as debugging or resuming chains.

# Sampling power (PolyChord settings).
nlive_value = int(2 * number_knots + 1)
num_repeats_value = int(2 * number_knots)

# Info dictionary for Cobaya.
info = {
    'debug': False,  # Allow to debug
    'likelihood': {'BOSSs': Pklike},  # Link likelihood with the previously defined class
    'theory': {
        'camb': {"external_primordial_pk": True},
        'my_pk': NodesInPrimordialPk
    },  # Include the primordial Pk with nodes in the theory class
    'params': {

        # Fixed cosmological parameters:
        'tau': tau_BOSS,
        'mnu': mnu_BOSS,
        'nnu': num_nu_massless_BOSS,
        'ombh2': Omega_b_BOSS_h2,
        'omch2': Omega_CDM_BOSS_h2,
        'H0': H0_BOSS,

        # Nuisance parameters with Gaussian or flat priors:
        'sigma_s': {
            'prior': {'dist': 'norm', 'loc': sigma_s_prior, 'scale': sigma_s_prior_std},
            'latex': 'sigma_s'
        },
        'sigma_nl': {
            'prior': {'dist': 'norm', 'loc': sigma_nl_prior, 'scale': sigma_nl_prior_std},
            'latex': 'sigma_nl'
        },
        'a1': {
            'prior': {'dist': 'norm', 'loc': a1_prior, 'scale': a1_prior_std},
            'latex': 'a_1'
        },
        'a2': {
            'prior': {'dist': 'norm', 'loc': a2_prior, 'scale': a2_prior_std},
            'latex': 'a_2'
        },
        'a3': {
            'prior': {'dist': 'norm', 'loc': a3_prior, 'scale': a3_prior_std},
            'latex': 'a_3'
        },
        'alpha': {
            'prior': {'min': alpha_prior - alpha_prior_std, 'max': alpha_prior + alpha_prior_std},
            'ref': alpha_prior,
            'latex': 'alpha'
        }
    },

    "sampler": {
        "polychord": {
            "nlive": nlive_value,
            "num_repeats": num_repeats_value,
            "precision_criterion": 1e-1
        }
    }
}

# Generate automatically the parameters x_i and y_i with priors.
# Loop over all nodes to define the parameters x_i and y_i.
for i in range(number_knots):

    # x1 (first node) and xN (last node) are fixed values, so for them we set constants.
    if i == 0:
        info['params']['x1'] = 0.0

    elif i == number_knots - 1:
        info['params'][f'x{number_knots}'] = 1.0

    # For the rest of parameters:
    # Define a flat prior between 0 and 1.
    else:
        info['params'][f'x{i+1}'] = {
            'prior': {'min': 0.0, 'max': 1.0},
            'ref': 0.5,
            'latex': f'x_{i+1}'
        }

    # The y_i have a flat prior in the range [-23, -17].
    info['params'][f'y{i+1}'] = {
        'prior': {'min': -23, 'max': -17},
        'ref': -20,
        'latex': f'y_{i+1}'
    }

# Prior of the multivariate Gaussian (cosmological).
# info["prior"] = {"Multivariate": lambda ombh2, omch2, H0, y1, y4:
#                  multivariate_gaussian_pdf(ombh2, omch2, H0, y1, y4)}

# Prior of the multivariate Gaussian of the model (nuisance parameters).
info["prior"] = {
    "Multivariate": lambda sigma_s, sigma_nl, a1, a2, a3:
        multivariate_gaussian_pdf_model(sigma_s, sigma_nl, a1, a2, a3)
}

# Path for output folder and name of the directory (dynamic according to number_knots).
info["output"] = (
    f"/Users/guillermo/Desktop/PruebaLeastSquares/"
    f"NGC_z1_BOSSPriors_{number_knots}Nodes_LeastSquaresResults"
)

In [42]:
# Execute Cobaya.
# Here, it will run in a Notebook, using the PolyChord sampler and the
# information contained in the `info` dictionary.

from cobaya import run
updated_info, sampler = run(info)

[output] Output to be read-from/written-into folder '/Users/guillermo/Desktop/PruebaLeastSquares', with prefix 'NGC_z1_BOSSPriors_4Nodes_LeastSquaresResults'
[prior] *WARNING* External prior 'Multivariate' loaded. Mind that it might not be normalized!
[camb] `camb` module loaded successfully from /Users/guillermo/Desktop/code/CAMB/camb
[polychord] `pypolychord` module loaded successfully from /Users/guillermo/Desktop/code/PolyChordLite/build/lib.macosx-10.9-universal2-3.9/pypolychord
[polychord] Storing raw PolyChord output in '/Users/guillermo/Desktop/PruebaLeastSquares/NGC_z1_BOSSPriors_4Nodes_LeastSquaresResults_polychord_raw'.
[model] Measuring speeds... (this may take a few seconds)
[0.000000000306541111157651 0.000000000237201061844661
 0.000000000270025650518198 0.000000000468666143798366
 0.000000000712643109904648 0.000000000997903068403776
 0.000000001322667640738697 0.000000001683973365136504
 0.000000002081467320461974 0.000000002513068749608266
 0.000000002978011029154245 

KeyboardInterrupt: 